# ROAD AI — Train YOLO11 Segmentation + Export ONNX
**Run each cell top to bottom. At the end you'll download `best.onnx`.**

Before starting: `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ── Cell 1 — Check GPU ──────────────────────────────────────────
!nvidia-smi

In [ ]:
# ── Cell 2 — Install dependencies ───────────────────────────────
!pip install ultralytics roboflow onnx onnxruntime -q

In [ ]:
# ── Cell 3 — Download your dataset from Roboflow ────────────────
# Your Roboflow credentials (already filled in from your project)
from roboflow import Roboflow

rf = Roboflow(api_key="T4zuK4T4cMUowqMPvSVl")
project = rf.workspace("cheemo").project("road-ai-segmentation")

# Download as YOLOv11 segmentation format
dataset = project.version(2).download("yolov11")

print(f"Dataset location: {dataset.location}")

In [ ]:
# ── Cell 4 — Check dataset structure ────────────────────────────
import os, yaml

data_yaml = os.path.join(dataset.location, "data.yaml")
with open(data_yaml) as f:
    cfg = yaml.safe_load(f)

print("Classes:", cfg.get("names"))
print("Num classes:", cfg.get("nc"))
print("Train path:", cfg.get("train"))
print("Val path:",   cfg.get("val"))

In [ ]:
# ── Cell 5 — Train YOLO11 segmentation ─────────────────────────
from ultralytics import YOLO

# yolo11n-seg  = nano  (fastest, use for Pi)
# yolo11s-seg  = small (better accuracy, still runs on Pi)
# yolo11m-seg  = medium (best accuracy, may be slow on Pi CPU)
model = YOLO("yolo11n-seg.pt")   # nano — best for Pi 4 CPU inference

results = model.train(
    data    = data_yaml,
    epochs  = 100,          # increase to 150 if you have time
    imgsz   = 640,
    batch   = 16,
    device  = 0,            # GPU
    patience= 20,           # early stop if no improvement for 20 epochs
    project = "roadai",
    name    = "yolo11n_seg",
    exist_ok= True,

    # Augmentation — matches your Roboflow preprocessing
    degrees  = 0.0,
    shear    = 14.0,        # ±14° as in your Roboflow config
    flipud   = 0.0,
    fliplr   = 0.5,
    blur     = 1.0,
)

print("Training complete!")

In [ ]:
# ── Cell 6 — Show results ───────────────────────────────────────
from IPython.display import Image as IPImage

results_dir = "roadai/yolo11n_seg"
IPImage(filename=f"{results_dir}/results.png", width=900)

In [ ]:
# ── Cell 7 — Print final metrics ────────────────────────────────
import pandas as pd

csv_path = f"{results_dir}/results.csv"
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()

last = df.iloc[-1]
print(f"mAP@50 (box):  {last.get('metrics/mAP50(B)', 'N/A')}")
print(f"mAP@50 (seg):  {last.get('metrics/mAP50(M)', 'N/A')}")
print(f"Precision:     {last.get('metrics/precision(B)', 'N/A')}")
print(f"Recall:        {last.get('metrics/recall(B)', 'N/A')}")

In [ ]:
# ── Cell 8 — Export best.pt → ONNX ─────────────────────────────
from ultralytics import YOLO

best_pt = f"{results_dir}/weights/best.pt"
model   = YOLO(best_pt)

# opset=12 for maximum compatibility with onnxruntime on Pi
model.export(
    format  = "onnx",
    imgsz   = 640,
    opset   = 12,
    dynamic = False,   # fixed batch size = 1 (Pi runs one frame at a time)
    simplify= True,    # shrinks the graph for faster runtime
)

onnx_path = best_pt.replace(".pt", ".onnx")
print(f"ONNX model saved to: {onnx_path}")

In [ ]:
# ── Cell 9 — Verify ONNX runs correctly ─────────────────────────
import onnxruntime as ort
import numpy as np

sess  = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
dummy = np.random.rand(1, 3, 640, 640).astype(np.float32)
out   = sess.run(None, {sess.get_inputs()[0].name: dummy})

print("ONNX verification passed!")
print(f"Output shapes: {[o.shape for o in out]}")
print(f"Input name:    {sess.get_inputs()[0].name}")

In [ ]:
# ── Cell 10 — Download both files to your computer ──────────────
from google.colab import files
import shutil

# Copy to root for easy download
shutil.copy(best_pt,   "best.pt")
shutil.copy(onnx_path, "best.onnx")

files.download("best.pt")    # PyTorch weights (backup)
files.download("best.onnx")  # ONNX for Pi inference

print("Done! Save best.onnx to road-ai/models/ on your laptop.")

## After downloading

1. Put `best.onnx` in `C:\Users\1nk\Desktop\road-ai\models\`
2. Copy it to the Pi: `scp models/best.onnx ubuntu@192.168.1.109:/home/ubuntu/roadai/`
3. Tell the dev (Claude) you have the file — next step is wiring it into roadai.py to replace the Roboflow API call